# Unsupervised Learning – Car Makes and Car Models with k-Means, Hierarchical Clustering and PCA

**Dataset:** Fuel consumption of 701 new car models of model year 2025 (Natural Resources Canada, Open Government
Licence – Canada): make, model, vehicle class, engine size, cylinders, transmission, fuel type and fuel consumption.
File: `../00_Data/car_fuel_consumption_2025.csv`.

**Questions:** Which car makes have a similar product range – makers of large SUVs and pickups, makers of efficient
everyday cars, premium and sports car makers? And what distinguishes one car model from another? There is **no target
variable** – we look for groups and for a compact description of the cars.

**After this exercise you can ...**
- perform a k-means iteration by hand,
- construct meaningful features for clustering,
- apply k-means (incl. scaling, elbow plot, silhouette analysis, several random starts) and interpret clusters,
- create and cut a dendrogram,
- use PCA to summarise many correlated features.

## Concept

The concepts behind this exercise (formulas and worked examples) are
explained in the notebook [clustering_pca_concept.ipynb](clustering_pca_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    adjusted_rand_score,
)
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-means by hand

*Time: about 15 minutes. Use the notebook cell below as your calculator: type the formulas with the numbers in
Python (e.g. `(3 + 5) / 2` or `0.4 ** 2`) – no paper needed.*

Six points in two dimensions: A (1, 1), B (2, 1), C (4, 3), D (5, 4), E (1, 2), F (5, 5).
We use $k = 2$ with the initial centres $\mu_1 = A = (1, 1)$ and $\mu_2 = C = (4, 3)$.

a) Compute the distances of all points to $\mu_2$ (the distances to $\mu_1$ are given) and assign every point to its
nearest centre.

b) Compute the two new centres: the mean of the $x_1$ values and the mean of the $x_2$ values of the points in each
cluster (e.g. `(1 + 2 + 1) / 3`).

c) Would any point change its cluster with the new centres? Has the algorithm converged?

In [ ]:
# Exercise 1: coordinates of the six points A, B, C, D, E, F
x1 = np.array([1, 2, 4, 5, 1, 5])
x2 = np.array([1, 1, 3, 4, 2, 5])

# a) Distances to the centres mu1 = (1, 1) and mu2 = (4, 3)
dist_1 = np.sqrt((x1 - 1) ** 2 + (x2 - 1) ** 2)
dist_2 = ___
print('Distances to mu1:', dist_1.round(2))
print('Distances to mu2:', dist_2.round(2))

# b) New centres (means of the points of each cluster)
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 2: Describe each car make by its product range

To cluster the car makes, we describe each make by a few features of its car models (means over all models of the make):

| feature | definition | high value means … |
|:---|:---|:---|
| `engine_size` | mean engine size [litres] | large engines |
| `consumption` | mean fuel consumption (`combined`) [L/100 km] | thirsty cars |
| `share_suv_pickup` | share of SUVs and pickups among the models | many large utility vehicles |
| `share_cvt` | share of models with a continuously variable transmission | many hybrids and efficient small cars |
| `city_highway_ratio` | mean city consumption / mean highway consumption | much more fuel in city traffic |

a) Import the car data. Compute the five features for each make (the code for the number of models and the first two
features is given). Keep only the makes with at least 5 models (with fewer models the means are unreliable).

b) Show the table sorted by `consumption` and describe the extremes.

In [ ]:
# a) Import the data as cars
#    File: '../00_Data/car_fuel_consumption_2025.csv'
# Your code here

# Helper columns: 1 = SUV or pickup, 1 = continuously variable transmission
cars['suv_pickup'] = (
    cars['vehicle_class'].str.contains('Sport utility|Pickup').astype(int)
)
cars['cvt'] = (cars['transmission_type'] == 'continuously variable').astype(int)

# Features per make (one row per make)
makes = pd.DataFrame()
makes['models'] = cars.groupby('make')['model'].count()
makes['engine_size'] = cars.groupby('make')['engine_size'].mean()
makes['consumption'] = cars.groupby('make')['combined'].mean()
# Your code here (share_suv_pickup, share_cvt, city_highway_ratio and
# keep the makes with at least 5 models)

# b) Table sorted by consumption
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 3: k-means – choosing k

a) Standardise the five features (not `models` – we want to group the makes by their *product range*, not by the number
of models).

b) Fit `KMeans(n_clusters=k, n_init=10, random_state=42)` for $k = 1, \dots, 8$ and plot the WCSS (`inertia_`) against
$k$ (elbow plot). Additionally compute the silhouette score for $k = 2, \dots, 8$.

c) Which $k$ do you choose?

In [ ]:
# Exercise 3: the five features
features = [
    'engine_size',
    'consumption',
    'share_suv_pickup',
    'share_cvt',
    'city_highway_ratio',
]

# a) Standardise
X = ___

# b) Elbow plot (k = 1..8) and silhouette score (k = 2..8)
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 4: Interpret the clusters

a) Fit k-means with $k = 3$ (`n_init=10, random_state=42`) and add the cluster labels to the table of the makes.

b) Compute the mean of each feature per cluster and list the makes of each cluster. Give each cluster a meaningful
name.

c) Plot the makes in a scatter plot of `engine_size` against `share_suv_pickup`, coloured by cluster, and write the name
of each make next to its point (the code for the names is given).

In [ ]:
# a) k-means with k = 3
km3 = ___
makes['cluster'] = ___

# b) Mean of each feature per cluster and makes of each cluster
# Your code here

# c) Figure showing the makes, coloured by cluster
plt.figure(figsize=(9, 6))
# Your code here (scatter plot)

# Names of the makes next to their points
for make in makes.index:
    plt.text(
        makes.loc[make, 'engine_size'],
        makes.loc[make, 'share_suv_pickup'],
        make,
        fontsize=8,
    )
plt.xlabel('Mean engine size [litres]')
plt.ylabel('Share of SUVs and pickups')
plt.show()

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 5: Silhouette analysis

The mean silhouette score of Exercise 3 summarises a clustering in one number. A **silhouette analysis** looks at the
silhouette value $s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$ of **every single make** ($a_i$ = mean distance to the
makes of its own cluster, $b_i$ = mean distance to the makes of the nearest other cluster):
$s_i$ close to 1 → well placed; close to 0 → between two clusters; negative → probably in the wrong cluster.

a) Compute the silhouette values of all makes with `silhouette_samples(X, labels)` for $k = 2, 3, 4$ and draw a
silhouette plot for each $k$: one horizontal bar per make, sorted within each cluster, with the names of the makes and a
vertical line at the mean silhouette score (the plotting code is given).

b) Which makes are badly placed for $k = 3$? Compare the three plots. Does the silhouette analysis support $k = 3$?

In [ ]:
# Function: draws one bar per make with its silhouette value, by cluster
def silhouette_plot(X, labels, names):
    """Silhouette plot: one bar per make, sorted within clusters."""
    values = silhouette_samples(X, labels)
    table = pd.DataFrame({'name': names, 'cluster': labels, 'sil': values})
    table = table.sort_values(['cluster', 'sil'])
    colors = plt.colormaps['tab10'](table['cluster'])
    plt.barh(table['name'], table['sil'], color=colors)
    plt.axvline(values.mean(), color='black', linestyle='--')
    plt.xlabel('silhouette value')
    plt.title(
        'k = '
        + str(len(np.unique(labels)))
        + ': mean = '
        + str(round(values.mean(), 2))
    )


# a) Silhouette plots for k = 2, 3 and 4 (one plot per k)
names = list(makes.index)
plt.figure(figsize=(18, 8))
for i, k in enumerate([2, 3, 4]):
    labels_k = ___
    plt.subplot(1, 3, i + 1)
    silhouette_plot(X, labels_k, names)
plt.tight_layout()
plt.show()

# b) Silhouette value of each make for k = 3
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 6: Stability of k-means

k-means depends on the random initial centres.

a) Fit k-means with $k = 3$ and `n_init=1` for `random_state` = 0, 1, …, 19. Print the WCSS of each run and compare the
cluster assignment with the solution of Exercise 4 using the *adjusted Rand index* (ARI: 1 = identical partition,
independent of the cluster numbers; ≈ 0 = random agreement).

b) What do you conclude? Why is `n_init=10` a good idea?

In [ ]:
# Exercise 6: k-means with n_init=1 for random_state = 0, ..., 19;
#    WCSS and ARI compared with km3
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 7: Hierarchical clustering and dendrogram

a) Compute the hierarchical clustering of the standardised features with `linkage(X, method='ward')` and draw the
dendrogram with the names of the makes as labels.

b) Cut the dendrogram into 3 clusters (`fcluster(Z, t=3, criterion='maxclust')`) and compare them with the k-means
clusters (`pd.crosstab`).

c) At which height would you cut the dendrogram? Which makes are merged last?

In [ ]:
# a) Ward linkage and dendrogram
Z = ___
# Your code here

# b) Cut into 3 clusters and compare with k-means
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 8: PCA – what makes car models different?

Now we turn the perspective around: the **701 car models** are the observations, five numerical features describe each
model: `engine_size`, `cylinders`, `city`, `highway` (fuel consumption in L/100 km) and `co2` (CO₂ emissions in g/km).

a) Standardise the five features and fit a PCA. Plot the explained variance ratio of the components. How much variance
does PC1 explain?

b) Show the loadings of PC1 and PC2 (`pca.components_`). Interpret both components.

c) Plot the car models in the PC1–PC2 plane, coloured by transmission (continuously variable or other). Which models
have the highest and the lowest PC2 scores?

In [ ]:
# a) The five numerical features of the 701 car models
pca_features = ['engine_size', 'cylinders', 'city', 'highway', 'co2']
Z_cars = StandardScaler().fit_transform(cars[pca_features])
pca = PCA()
scores = ___
# Your code here

# b) Loadings of PC1 and PC2 for each feature
# Your code here

# c) Figure showing the car models in the PC1-PC2 plane
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')